# FM-MCAN-HDD: Trust-Aware Multimodal Privacy-Preserving Federated Learning

**Notebook scope:** runnable research prototype inspired by the uploaded paper *“Trust-Aware Multimodal Privacy-Preserving Federated Learning Framework with Cross-Modal Attention for Early Heart Disease Diagnosis.”*

This notebook runs end-to-end on CPU using **clearly labelled synthetic data** so the full pipeline can be tested without downloading private medical datasets. It implements:
- Four simulated hospital clients with heterogeneous modality availability
- Modality-specific encoders and cross-modal attention
- Trust-weighted federated aggregation
- Gradient/update clipping and Gaussian noise as a **DP-inspired prototype** (not a formal privacy guarantee)
- Local evaluation, confusion matrix, ROC-AUC, training curves
- Simple local feature-importance explanations and saved artifacts

**Important research limitations:** synthetic results are not clinical evidence and will not reproduce the paper's reported metrics. The paper describes CT/MRI, ECG chart images, raw ECG, and clinical records, but those sources do not automatically share patient identifiers. Real patient-level multimodal fusion requires valid cross-modal linkage and consistent labels. This notebook uses a controlled synthetic aligned test set and modality-siloed simulated clients. The aggregation code averages a common shared model; it is not production homomorphic encryption. Do not use for diagnosis.

## 1. Configuration and imports

Run the notebook from top to bottom. For real datasets, replace the synthetic data-generation cell and adapt each local loader to your dataset schema. Keep patient-level train/validation/test separation to prevent leakage.

In [ ]:
import os, copy, json, random, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve
)
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path("FM_MCAN_HDD_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_CLIENTS = 4
N_CLASSES = 2  # binary: 0 = no disease, 1 = disease
MODALITIES = ["cardiac_image", "ecg_chart", "raw_ecg", "clinical"]
FEATURE_DIM = 32
LATENT_DIM = 64
BATCH_SIZE = 32
LOCAL_EPOCHS = 2
COMMUNICATION_ROUNDS = 8
LEARNING_RATE = 5e-4
TRUST_TEMPERATURE = 0.8
CLIP_NORM = 1.5
DP_NOISE_MULTIPLIER = 0.05  # demonstration only; tune and account formally for real DP
print("Device:", DEVICE)
print("Outputs:", OUT_DIR.resolve())

## 2. Create a reproducible synthetic multimodal benchmark

Each simulated patient has three 32-dimensional feature vectors (image, ECG, clinical) and an availability mask. Four simulated hospitals are then built as modality silos: each hospital sees only its assigned modality. The target is binary disease status. **These are synthetic features, not real patient measurements.**

In [ ]:
def make_synthetic_multimodal(n=1200, seed=SEED):
    rng = np.random.default_rng(seed)
    y = rng.binomial(1, 0.48, size=n).astype(np.int64)
    # Latent disease signal plus modality-specific noise
    base = rng.normal(0, 1, size=(n, FEATURE_DIM)).astype(np.float32)
    direction = rng.normal(0, 1, size=FEATURE_DIM).astype(np.float32)
    direction /= (np.linalg.norm(direction) + 1e-8)
    signal = (2*y - 1).astype(np.float32)[:, None] * direction[None, :] * 1.25
    image = base + signal + rng.normal(0, .75, (n, FEATURE_DIM)).astype(np.float32)
    ecg = 0.7*base + 1.15*signal + rng.normal(0, .70, (n, FEATURE_DIM)).astype(np.float32)
    clinical = 0.4*base + 1.35*signal + rng.normal(0, .65, (n, FEATURE_DIM)).astype(np.float32)
    return {
        "image": image.astype(np.float32),
        "ecg": ecg.astype(np.float32),
        "clinical": clinical.astype(np.float32),
        "y": y
    }

data = make_synthetic_multimodal()
idx = np.arange(len(data["y"]))
train_idx, test_idx = train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=data["y"]
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.125, random_state=SEED, stratify=data["y"][train_idx]
)
print(f"Train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}")
print("Synthetic class distribution:", pd.Series(data["y"]).value_counts().sort_index().to_dict())

# Normalize using training data only; apply the same scaler to validation/test.
scalers = {}
for key in ["image", "ecg", "clinical"]:
    scaler = StandardScaler()
    scaler.fit(data[key][train_idx])
    data[key] = scaler.transform(data[key]).astype(np.float32)
    scalers[key] = scaler

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "samples": [len(train_idx), len(val_idx), len(test_idx)]
}).to_csv(OUT_DIR / "split_summary.csv", index=False)

## 3. Build the four local hospital datasets

Client A receives image-derived features, Client B ECG-chart-derived features, Client C raw-ECG-derived features, and Client D clinical features. The model input shape stays consistent so the shared parameters can be federated; unavailable modalities are zero-filled and accompanied by an availability mask.

In [ ]:
def build_client_arrays(indices, client_id):
    # All model inputs have shape (N, 3, FEATURE_DIM); only the assigned modality is available.
    X = np.zeros((len(indices), 3, FEATURE_DIM), dtype=np.float32)
    mask = np.zeros((len(indices), 3), dtype=bool)
    if client_id == 0:  # cardiac CT/MRI feature proxy
        X[:, 0, :] = data["image"][indices]; mask[:, 0] = True
    elif client_id == 1:  # ECG chart image feature proxy
        X[:, 1, :] = data["ecg"][indices]; mask[:, 1] = True
    elif client_id == 2:  # raw ECG feature proxy (same synthetic ECG source here)
        X[:, 1, :] = data["ecg"][indices]; mask[:, 1] = True
    elif client_id == 3:  # structured clinical feature proxy
        X[:, 2, :] = data["clinical"][indices]; mask[:, 2] = True
    return torch.tensor(X), torch.tensor(mask), torch.tensor(data["y"][indices], dtype=torch.long)

clients_train = [build_client_arrays(train_idx, k) for k in range(N_CLIENTS)]
clients_val = [build_client_arrays(val_idx, k) for k in range(N_CLIENTS)]
# Multimodal evaluation uses all three modalities on the held-out synthetic test set.
X_test = np.stack([data["image"][test_idx], data["ecg"][test_idx], data["clinical"][test_idx]], axis=1)
M_test = np.ones((len(test_idx), 3), dtype=bool)
y_test = data["y"][test_idx]
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(M_test), torch.tensor(y_test, dtype=torch.long))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print("Client training sizes:", [len(c[2]) for c in clients_train])

## 4. Model: modality encoders + cross-modal attention + diagnosis head

Each modality has a small encoder. The encoded modality tokens are processed by multi-head self-attention with an availability mask, then pooled into a binary diagnosis head. This is a lightweight executable approximation of the paper's modality-specific encoders and cross-modal attention, not a literal 3D Swin/DenseNet CVTF-Net or full TCANet implementation.

In [ ]:
class FMCANHDD(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM, latent_dim=LATENT_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.image_encoder = nn.Sequential(
            nn.Linear(feature_dim, latent_dim), nn.GELU(), nn.LayerNorm(latent_dim), nn.Dropout(0.15)
        )
        self.ecg_encoder = nn.Sequential(
            nn.Linear(feature_dim, latent_dim), nn.GELU(), nn.LayerNorm(latent_dim), nn.Dropout(0.15)
        )
        self.clinical_encoder = nn.Sequential(
            nn.Linear(feature_dim, latent_dim), nn.GELU(), nn.LayerNorm(latent_dim), nn.Dropout(0.15)
        )
        self.attn = nn.MultiheadAttention(latent_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(latent_dim)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.GELU(), nn.Dropout(0.15), nn.Linear(32, n_classes)
        )

    def forward(self, x, availability, return_attention=False):
        # x: [batch, 3, feature_dim], availability: [batch, 3], True means present
        tokens = torch.stack([
            self.image_encoder(x[:, 0, :]),
            self.ecg_encoder(x[:, 1, :]),
            self.clinical_encoder(x[:, 2, :])
        ], dim=1)
        # MHA key_padding_mask=True means ignore token; protect against all-missing rows.
        missing = ~availability.bool()
        all_missing = missing.all(dim=1)
        if all_missing.any():
            missing = missing.clone()
            missing[all_missing, 0] = False
        attended, weights = self.attn(
            tokens, tokens, tokens, key_padding_mask=missing, need_weights=True, average_attn_weights=True
        )
        fused_tokens = self.norm(tokens + attended)
        valid = availability.float().unsqueeze(-1)
        pooled = (fused_tokens * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        logits = self.classifier(pooled)
        if return_attention:
            return logits, weights
        return logits

model = FMCANHDD().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Local training, trust scoring, and trust-weighted aggregation

Each hospital trains locally and sends model updates—not raw examples—to the simulated server. Trust scores combine local validation accuracy and a small floor. The server weights client states using a softmax over trust scores, clips the aggregated update, and optionally adds Gaussian noise. This is a **simulation**: no encrypted transport or Paillier homomorphic encryption is implemented, and the noise setting alone does not establish an ε-DP guarantee.

In [ ]:
def make_loader(arrays, shuffle=True):
    X, mask, y = arrays
    ds = TensorDataset(X, mask, y)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def evaluate_model(model, loader):
    model.eval()
    ys, probs, preds, losses = [], [], [], []
    with torch.no_grad():
        for X, mask, y in loader:
            X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            logits = model(X, mask)
            loss = F.cross_entropy(logits, y)
            p = torch.softmax(logits, dim=1)[:, 1]
            ys.extend(y.cpu().numpy().tolist())
            probs.extend(p.cpu().numpy().tolist())
            preds.extend(logits.argmax(1).cpu().numpy().tolist())
            losses.append(loss.item())
    auc = roc_auc_score(ys, probs) if len(set(ys)) == 2 else float("nan")
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(ys, preds),
        "precision": precision_score(ys, preds, zero_division=0),
        "recall": recall_score(ys, preds, zero_division=0),
        "f1": f1_score(ys, preds, zero_division=0),
        "auc": auc,
        "y_true": np.asarray(ys), "y_prob": np.asarray(probs), "y_pred": np.asarray(preds)
    }

def train_local(global_state, train_arrays, val_arrays, epochs=LOCAL_EPOCHS):
    local_model = FMCANHDD().to(DEVICE)
    local_model.load_state_dict(global_state)
    optimizer = torch.optim.AdamW(local_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    train_loader = make_loader(train_arrays, shuffle=True)
    val_loader = make_loader(val_arrays, shuffle=False)
    for _ in range(epochs):
        local_model.train()
        for X, mask, y in train_loader:
            X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = local_model(X, mask)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), CLIP_NORM)
            optimizer.step()
    val_metrics = evaluate_model(local_model, val_loader)
    return copy.deepcopy(local_model.state_dict()), val_metrics

def trust_weighted_aggregate(global_state, client_states, trust_scores, noise_multiplier=DP_NOISE_MULTIPLIER):
    scores = torch.tensor(trust_scores, dtype=torch.float32)
    weights = torch.softmax(scores / TRUST_TEMPERATURE, dim=0).cpu().numpy()
    new_state = {}
    for key in global_state:
        base = global_state[key].detach().cpu()
        # Aggregate deltas relative to the current global model.
        delta = torch.zeros_like(base)
        for w, state in zip(weights, client_states):
            delta += float(w) * (state[key].detach().cpu() - base)
        # Clip each tensor's aggregate delta and add demonstration Gaussian noise.
        norm = torch.linalg.vector_norm(delta.float())
        if norm > CLIP_NORM:
            delta = delta * (CLIP_NORM / (norm + 1e-12))
        if noise_multiplier > 0 and torch.is_floating_point(delta):
            delta = delta + torch.randn_like(delta) * (noise_multiplier * CLIP_NORM)
        new_state[key] = (base + delta).to(global_state[key].device)
    return new_state, weights

global_model = FMCANHDD().to(DEVICE)
global_state = copy.deepcopy(global_model.state_dict())
history = []
round_client_weights = []

for rnd in range(1, COMMUNICATION_ROUNDS + 1):
    local_states, val_metrics, trust_scores = [], [], []
    for client_id in range(N_CLIENTS):
        state, metrics = train_local(global_state, clients_train[client_id], clients_val[client_id])
        local_states.append(state)
        val_metrics.append(metrics)
        # Reliability proxy: validation accuracy; real deployments need audited quality/reliability signals.
        trust_scores.append(float(np.clip(metrics["accuracy"], 0.05, 0.99)))
    global_state, weights = trust_weighted_aggregate(
        global_state, local_states, trust_scores, noise_multiplier=DP_NOISE_MULTIPLIER
    )
    global_model.load_state_dict(global_state)
    round_client_weights.append(weights)
    multimodal_metrics = evaluate_model(global_model, test_loader)
    history.append({
        "round": rnd,
        "test_accuracy": multimodal_metrics["accuracy"],
        "test_precision": multimodal_metrics["precision"],
        "test_recall": multimodal_metrics["recall"],
        "test_f1": multimodal_metrics["f1"],
        "test_auc": multimodal_metrics["auc"],
        "mean_client_val_accuracy": float(np.mean([m["accuracy"] for m in val_metrics]))
    })
    print(f"Round {rnd:02d}/{COMMUNICATION_ROUNDS} | "
          f"test acc={multimodal_metrics['accuracy']:.3f} | "
          f"F1={multimodal_metrics['f1']:.3f} | AUC={multimodal_metrics['auc']:.3f} | "
          f"client weights={np.round(weights, 3)}")

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "federated_training_history.csv", index=False)
pd.DataFrame(round_client_weights, columns=[f"Client_{i+1}" for i in range(N_CLIENTS)]).to_csv(
    OUT_DIR / "client_aggregation_weights.csv", index=False
)

## 6. Final evaluation and plots

Metrics are computed on the held-out synthetic multimodal test split. For clinical experiments, evaluate on real held-out patients and report confidence intervals, class-wise results, calibration, and external-site validation.

In [ ]:
final = evaluate_model(global_model, test_loader)
metric_table = pd.DataFrame([{
    "Accuracy (%)": 100*final["accuracy"],
    "Precision (%)": 100*final["precision"],
    "Recall (%)": 100*final["recall"],
    "F1-score (%)": 100*final["f1"],
    "ROC-AUC (%)": 100*final["auc"]
}]).round(2)
display(metric_table)
metric_table.to_csv(OUT_DIR / "final_test_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history_df["round"], history_df["test_accuracy"]*100, marker="o", label="Test accuracy")
ax.plot(history_df["round"], history_df["test_f1"]*100, marker="s", label="Test F1-score")
ax.set_xlabel("Federated communication round")
ax.set_ylabel("Score (%)")
ax.set_title("FM-MCAN-HDD Synthetic Benchmark: Federated Performance")
ax.grid(True, alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "federated_performance.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(final["y_true"], final["y_pred"], labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["No disease", "Disease"]).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix — Synthetic Test Data")
fig.tight_layout(); fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=300); plt.show()

fpr, tpr, _ = roc_curve(final["y_true"], final["y_prob"])
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"AUC = {final['auc']:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC Curve — Synthetic Test Data")
ax.grid(True, alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "roc_curve.png", dpi=300); plt.show()

print(classification_report(final["y_true"], final["y_pred"], target_names=["No disease", "Disease"], zero_division=0))


## 7. Attention inspection and local feature attribution

The following provides a lightweight model explanation for one synthetic test sample: attention weights across modality tokens and input-feature perturbation importance. It is a prototype, not a clinical explanation. For real image inputs, use Grad-CAM on the actual CNN feature maps; for structured clinical data, SHAP can be added after validating the tabular pipeline.

In [ ]:
global_model.eval()
sample_x = torch.tensor(X_test[:1], dtype=torch.float32, device=DEVICE)
sample_mask = torch.tensor(M_test[:1], dtype=torch.bool, device=DEVICE)
with torch.no_grad():
    logits, attn = global_model(sample_x, sample_mask, return_attention=True)
    prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    attn_values = attn[0].cpu().numpy()

print("Predicted class:", int(np.argmax(prob)), "(0=no disease, 1=disease)")
print("Class probabilities:", np.round(prob, 4))
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(attn_values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(3), ["Image", "ECG", "Clinical"])
ax.set_yticks(range(3), ["Image", "ECG", "Clinical"])
ax.set_title("Cross-Modal Attention Weights (Synthetic Example)")
fig.colorbar(im, ax=ax, label="Attention weight")
fig.tight_layout(); fig.savefig(OUT_DIR / "cross_modal_attention.png", dpi=300); plt.show()

# Permutation/ablation attribution: change one feature dimension at a time to its training mean (0 after scaling).
x0 = sample_x.clone()
with torch.no_grad():
    p0 = torch.softmax(global_model(x0, sample_mask), dim=1)[0, 1].item()
importance = []
for modality_idx, modality_name in enumerate(["Image", "ECG", "Clinical"]):
    for feature_idx in range(FEATURE_DIM):
        xp = x0.clone()
        xp[0, modality_idx, feature_idx] = 0.0
        with torch.no_grad():
            pp = torch.softmax(global_model(xp, sample_mask), dim=1)[0, 1].item()
        importance.append((modality_name, feature_idx, abs(p0-pp)))
importance_df = pd.DataFrame(importance, columns=["Modality", "Feature_Index", "Probability_Change"])
top_importance = importance_df.sort_values("Probability_Change", ascending=False).head(15)
display(top_importance)
top_importance.to_csv(OUT_DIR / "top_feature_attributions.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_df = top_importance.copy()
plot_df["Feature"] = plot_df["Modality"] + "_" + plot_df["Feature_Index"].astype(str)
ax.bar(plot_df["Feature"], plot_df["Probability_Change"])
ax.set_ylabel("Absolute change in disease probability")
ax.set_title("Top Feature Perturbation Attributions")
ax.tick_params(axis="x", rotation=60)
fig.tight_layout(); fig.savefig(OUT_DIR / "feature_attributions.png", dpi=300); plt.show()

## 8. Save model and run metadata

The saved checkpoint is for this synthetic prototype. Keep a separate data manifest and record exact dataset versions, patient-level splits, preprocessing parameters, random seeds, and privacy accounting when moving to real experiments.

In [ ]:
checkpoint_path = OUT_DIR / "fm_mcan_hdd_synthetic_checkpoint.pt"
torch.save({
    "model_state_dict": global_model.state_dict(),
    "config": {
        "feature_dim": FEATURE_DIM, "latent_dim": LATENT_DIM, "n_classes": N_CLASSES,
        "n_clients": N_CLIENTS, "communication_rounds": COMMUNICATION_ROUNDS,
        "local_epochs": LOCAL_EPOCHS, "clip_norm": CLIP_NORM,
        "dp_noise_multiplier_demo_only": DP_NOISE_MULTIPLIER, "seed": SEED
    },
    "note": "Trained on synthetic demonstration data; not a clinical model."
}, checkpoint_path)

metadata = {
    "project": "FM-MCAN-HDD",
    "data_status": "synthetic demonstration data",
    "device": str(DEVICE),
    "seed": SEED,
    "train_samples": int(len(train_idx)),
    "validation_samples": int(len(val_idx)),
    "test_samples": int(len(test_idx)),
    "final_metrics": {k: float(v) for k, v in metric_table.iloc[0].to_dict().items()},
    "limitations": [
        "No real patient data used",
        "No production homomorphic encryption",
        "Differential privacy is not formally accounted",
        "Clinical safety and external validation not established"
    ]
}
with open(OUT_DIR / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved checkpoint:", checkpoint_path)
print("Output files:")
for path in sorted(OUT_DIR.iterdir()):
    print(" -", path.name)

## 9. Adapting this notebook to real data

1. **Cardiac CT/MRI:** load image volumes/slices and implement the paper's anisotropic diffusion, CLAHE, segmentation/registration, then a proper image encoder. This prototype accepts precomputed 32-D features rather than raw DICOM/PNG.
2. **ECG chart images:** load labelled ECG images and replace the synthetic ECG feature array with image embeddings from a trained encoder.
3. **Raw ECG:** load signal arrays/CSV; apply 0.5–40 Hz zero-phase Butterworth filtering, R-peak/beat segmentation, normalization, and a 1-D CNN/BiLSTM temporal encoder.
4. **Clinical records:** map the actual target and feature columns; split by patient before fitting imputers/scalers; fit all preprocessing on training data only.
5. **Multimodal alignment:** do not pair unrelated records by row number. Use verified patient IDs and consented linkage, or evaluate modality-siloed learning separately.
6. **Federated deployment:** run each client process at its institution, use an audited Flower strategy, authenticated TLS, a vetted secure aggregation implementation, and formal differential-privacy accounting. The current notebook simulates all clients in one process.
7. **Evaluation:** do not force the manuscript's reported accuracy/AUC into the results. Compute them from held-out real predictions, preferably with patient-level and site-level splits.

This notebook is an implementation scaffold, not a certified medical device.